# HW5: Evaluate Before You Automate

**COMPSS 211A | Fall 2026 | Student copy**

Researchers use LLMs to label text so they don't have to read all of it. Before you use such labels, you need to check them. In this homework you ask Gemini to predict the verdict on 40 r/AmItheAsshole posts, compare its answers with the community's verdicts, and read the posts it got wrong.

**Due:** Sunday, November 22, 2026 at 11:59 p.m.

Fill in every cell marked "Your turn" and answer the written questions. When a cell asks for a prediction, write your guess before you run anything. Keep the variable names we ask for, because we look for them when grading.

## The data

You used these r/AmItheAsshole posts in HW4: 5,000 of the most upvoted posts from 2018 to 2021. Each post has a verdict from its top-voted comment:

| Verdict | Meaning |
| --- | --- |
| NTA | The poster was not in the wrong. |
| YTA | "You're the asshole": the poster was in the wrong. |
| ESH | Everyone involved behaved badly. |
| NAH | Nobody behaved badly. |

This time you ask whether an LLM can predict the verdict. The verdict is the result of a vote, not a fact about who was right. These are real people's stories: don't try to find out who wrote them, and don't quote them outside this class.

## What you need

- Your Gemini API key in the `.env` file in the top folder of the course repository. See "API keys" in `SETUP.md`. Never paste the key into the notebook.
- Run the first cell of Task 2 early. It stops with a message if it can't find your key.
- The 40 calls take about five minutes and cost a few cents at most. The results are saved to a file, so you only call Gemini once.

If your key doesn't work, ask on Slack or come to office hours before the deadline.

## What you will practice

- Setting up baselines before judging a model.
- Reading a system instruction, a prompt template, and a schema (Week 7).
- Scoring predictions with accuracy and a confusion table (Week 10).
- Reading a model's mistakes.

## AI and collaboration policy

Task 2 uses Gemini. Beyond that, you may use AI to help you understand an error message, as long as you say so in a note in your notebook. Write the code and the written answers yourself.

In [ ]:
from pathlib import Path
import json
import os
import sys
import time

import pandas as pd
from IPython.display import display
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

SUPPORTED_PYTHON = (3, 13)
if sys.version_info[:2] != SUPPORTED_PYTHON:
    print(
        "Setup note: this course is tested with Python 3.13; "
        f"you are running {sys.version.split()[0]}."
    )

def find_course_root():
    """Find the cloned repository when this notebook is running locally."""
    for folder in (Path.cwd(), *Path.cwd().parents):
        if (folder / "pyproject.toml").exists() and (folder / "data").is_dir():
            return folder
    return None

LOCAL_COURSE_ROOT = find_course_root()
COURSE_ROOT = LOCAL_COURSE_ROOT or Path.cwd()
DATA_DIR = (
    LOCAL_COURSE_ROOT / "data"
    if LOCAL_COURSE_ROOT
    else COURSE_ROOT / "compss211_data"
)
GENERATED_DIR = COURSE_ROOT / "generated"
DATA_DIR.mkdir(parents=True, exist_ok=True)
GENERATED_DIR.mkdir(parents=True, exist_ok=True)

DATA_BASE_URL = (
    "https://raw.githubusercontent.com/"
    "macss-berkeley/compss-211a/main/data"
)

def course_data_path(filename):
    """Use a local course file, or download it when running in Colab."""
    path = DATA_DIR / filename
    if not path.exists():
        from urllib.request import urlretrieve

        urlretrieve(f"{DATA_BASE_URL}/{filename}", path)
    return path

print(f"Python {sys.version.split()[0]} | data={DATA_DIR}")

## 1. Start with baselines

Before you judge a model, check how well you could do without it. The simplest baseline always guesses the most common verdict, which here is NTA.

The cell below loads the posts (with the same filter as HW4) and picks 10 posts of each verdict, 40 in total. Everyone gets the same 40 posts because `random_state` is fixed. The other posts are used to train the classifier in Task 3.

**Your turn:**

1. Make `majority_accuracy_all`: the accuracy of always guessing NTA on all posts (hint: `(posts["verdict"] == "NTA").mean()`).
2. Make `majority_accuracy_sample`: the same for `evaluation_sample`.
3. In a comment, explain why the two numbers are different.
4. In a comment, write down the accuracy you expect Gemini to get on the 40 posts.

In [ ]:
VERDICT_LABELS = {
    "Not the A-hole": "NTA",
    "not the a-hole": "NTA",
    "Asshole": "YTA",
    "Everyone Sucks": "ESH",
    "No A-holes here": "NAH",
}
ALLOWED_LABELS = {"NTA", "YTA", "ESH", "NAH"}

raw_posts = pd.read_csv(course_data_path("aita_top_subs.csv"))
keep = (
    raw_posts["flair_text"].isin(VERDICT_LABELS)
    & raw_posts["selftext"].notna()
    & ~raw_posts["selftext"].isin(["[removed]", "[deleted]"])
)
posts = raw_posts.loc[keep, ["idstr", "title", "selftext", "flair_text"]].copy()
posts["verdict"] = posts["flair_text"].map(VERDICT_LABELS)

evaluation_sample = posts.groupby("verdict").sample(n=10, random_state=211)
training_posts = posts.drop(evaluation_sample.index)

print(len(posts), "posts;", len(evaluation_sample), "in the evaluation sample;", len(training_posts), "for training")
posts["verdict"].value_counts()

In [ ]:
# Your turn
majority_accuracy_all = ...
majority_accuracy_sample = ...

# Why the two numbers are different:
#

# My prediction for Gemini's accuracy on the 40 posts:
#

## 2. Ask Gemini

The code for this task is supplied. Read it, then run the cells in order.

1. The first cell connects to Gemini with your key.
2. The second cell holds what Gemini is told: a system instruction with the task and the four labels, a prompt template with slots for the title and text, and a schema that limits the reply to one label and a reason. These are the same parts as in the Week 7 workspace.
3. The third cell has two functions. `ask_gemini` sends one prompt and returns the label and the reason. `label_posts` calls it for each post.
4. The last cell labels the 40 posts and saves the results to `generated/hw5_gemini_labels.csv`. If the file already exists, it loads it instead of calling Gemini again.

In [ ]:
# Supplied: connect to Gemini. The key is read from your .env file and never printed.
from dotenv import load_dotenv
from google import genai
from google.genai import types

load_dotenv(COURSE_ROOT / ".env")
key = os.environ.get("GEMINI_API_KEY", "")
if not key or key.startswith("paste-"):
    raise ValueError("Add GEMINI_API_KEY to your .env file, then run this cell again.")

client = genai.Client(api_key=key)
GEMINI_MODEL = "gemini-3.5-flash-lite"  # use the replacement model if bCourses announces one
print("Connected to Gemini.")

In [ ]:
# Supplied: what Gemini is told for every post.
SYSTEM = """You read posts from the Reddit forum r/AmItheAsshole, where people describe a conflict and ask whether they were in the wrong.

Predict the verdict the Reddit community gave. Use one of these labels:
NTA (not the asshole), YTA (you're the asshole), ESH (everyone sucks here), NAH (no assholes here).

Also give a reason of one sentence."""

PROMPT_TEMPLATE = """Title: {title}

{text}"""

# The schema says what Gemini must send back: one of the four labels, and a reason.
LABEL_SCHEMA = {
    "type": "object",
    "properties": {
        "llm_label": {"type": "string", "enum": sorted(ALLOWED_LABELS)},
        "llm_reason": {"type": "string"},
    },
    "required": ["llm_label", "llm_reason"],
}

In [ ]:
# Supplied: send one post, and label every post in a sample.
def ask_gemini(prompt):
    """Send one prompt. Return the label and the reason."""
    response = client.models.generate_content(
        model=GEMINI_MODEL,
        contents=prompt,
        config=types.GenerateContentConfig(
            system_instruction=SYSTEM,
            response_mime_type="application/json",
            response_schema=LABEL_SCHEMA,
        ),
    )
    time.sleep(6)  # wait between calls to stay under the free tier's limit per minute
    return json.loads(response.text)


def label_posts(sample):
    """Ask Gemini about every post in sample. Return one row per post."""
    rows = []
    for post in sample.itertuples():
        prompt = PROMPT_TEMPLATE.format(title=post.title, text=post.selftext)
        reply = ask_gemini(prompt)
        rows.append({"idstr": post.idstr, "verdict": post.verdict, **reply})
        print(len(rows), "of", len(sample), "done")
    return pd.DataFrame(rows)

In [ ]:
# Supplied: label the 40 posts once and save the results. If the file already exists, load it instead.
LABELS_PATH = GENERATED_DIR / "hw5_gemini_labels.csv"

if LABELS_PATH.exists():
    live_results = pd.read_csv(LABELS_PATH)
else:
    live_results = label_posts(evaluation_sample)
    live_results["model"] = GEMINI_MODEL
    live_results.to_csv(LABELS_PATH, index=False)

live_results[["idstr", "verdict", "llm_label", "llm_reason"]].head()

## 3. Score the predictions

You will compare three methods on the same 40 posts:

- Always NTA: the baseline from Task 1.
- Traditional: a TF-IDF and logistic regression classifier like the one in Week 10, trained on the other posts. The cell below builds it. `class_weight="balanced"` stops it from guessing NTA every time.
- LLM: Gemini's labels from Task 2.

**Your turn:**

1. Add a column `traditional_prediction` to `live_results` (hint: `merge` with `evaluation_sample[["idstr", "traditional_prediction"]]` on `idstr`).
2. Make `evaluation_summary`: a table with the columns `method` and `accuracy`, and one row for each method.
3. Make `confusion_table` for Gemini with `pd.crosstab(live_results["verdict"], live_results["llm_label"])`. Rows are the community's verdict and columns are Gemini's.
4. Make `accuracy_by_verdict`: Gemini's accuracy for each verdict (hint: make a True/False column, then use `groupby("verdict")`).

Gemini doesn't always give the same answer to the same prompt, so your numbers may differ a little from your classmates'.

In [ ]:
baseline_vectorizer = TfidfVectorizer(stop_words="english", min_df=5)
training_matrix = baseline_vectorizer.fit_transform(training_posts["title"] + " " + training_posts["selftext"])
baseline_model = LogisticRegression(max_iter=1000, class_weight="balanced")
baseline_model.fit(training_matrix, training_posts["verdict"])

evaluation_matrix = baseline_vectorizer.transform(evaluation_sample["title"] + " " + evaluation_sample["selftext"])
evaluation_sample["traditional_prediction"] = baseline_model.predict(evaluation_matrix)
evaluation_sample["traditional_prediction"].value_counts()

In [ ]:
# Your turn 1: traditional_prediction


# Your turn 2: evaluation_summary
evaluation_summary = ...

# Your turn 3: confusion_table
confusion_table = ...

# Your turn 4: accuracy_by_verdict
accuracy_by_verdict = ...


## 4. Read the mistakes

**Your turn:**

1. Make `llm_errors`: the rows of `live_results` where Gemini's label is different from the community's verdict. Add the post `title` by merging with `evaluation_sample`. Keep `idstr`, `title`, `verdict`, `llm_label`, and `llm_reason`.
2. Read at least three mistakes, including the full post (`evaluation_sample.loc[evaluation_sample["idstr"] == "...", "selftext"].iloc[0]`). Fill in the table below. If you think Gemini was right, say so: the community's verdict is a vote, not a fact.

In [ ]:
# Your turn 1: llm_errors
llm_errors = ...

# Your turn 2: print the full text of the posts you read


### Your notes on three mistakes

| idstr | Community | Gemini | Why I think Gemini disagreed | Who do I agree with? |
| --- | --- | --- | --- | --- |
|  |  |  |  |  |
|  |  |  |  |  |
|  |  |  |  |  |

### Your response


Answer in three or four sentences:

1. How did Gemini compare with the two baselines? Which verdicts did it get right, and which did it miss?
2. The community's verdict is a Reddit vote. What does that mean for how you read Gemini's accuracy?
3. Would you use Gemini to label all the posts? What would you check first?


> Write your response here, then delete this line.